In [0]:
# ==========================================
# EXPLORACIÓN DEL ARCHIVO DE DEVOLUCIONES
# ==========================================

RUTA_DEVOLUCIONES = "/Volumes/proyecto_final/default/vol_landing/devoluciones"

ARCHIVO = f"{RUTA_DEVOLUCIONES}/devoluciones_lote_01_2026-05-31.csv"

# Leer el archivo CSV
df_devoluciones = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ARCHIVO)
)

# Mostrar estructura
print("ESQUEMA DEL ARCHIVO:")
df_devoluciones.printSchema()

# Contar registros
print("TOTAL DE REGISTROS:", df_devoluciones.count())

# Mostrar una muestra
display(df_devoluciones.limit(10))

In [0]:
# ==========================================
# CONFIGURACIÓN AUTO LOADER - DEVOLUCIONES
# ==========================================

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp, col

CATALOGO = "proyecto_final"

RUTA_ORIGEN = "/Volumes/proyecto_final/default/vol_landing/devoluciones"

RUTA_CHECKPOINT = (
    "/Volumes/proyecto_final/default/vol_landing/"
    "_checkpoints/devoluciones"
)

TABLA_DESTINO = f"{CATALOGO}.bronze.devoluciones"

# Conservar los datos originales como STRING
ESQUEMA_DEVOLUCIONES = StructType([
    StructField("devolucion_id", StringType(), True),
    StructField("venta_id", StringType(), True),
    StructField("sucursal_id", StringType(), True),
    StructField("producto_id", StringType(), True),
    StructField("motivo", StringType(), True),
    StructField("monto_reembolso", StringType(), True),
    StructField("fecha_devolucion", StringType(), True)
])

print("Origen:", RUTA_ORIGEN)
print("Checkpoint:", RUTA_CHECKPOINT)
print("Destino:", TABLA_DESTINO)

print("ESQUEMA CONFIGURADO:")
print(ESQUEMA_DEVOLUCIONES.simpleString())

In [0]:
# ==========================================
# INGESTA BRONZE - DEVOLUCIONES
# ==========================================

# Lectura incremental de archivos CSV
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(ESQUEMA_DEVOLUCIONES)
    .load(RUTA_ORIGEN)
)

# Agregar campos de trazabilidad
df_bronze = (
    df_stream
    .withColumn("archivo_origen", col("_metadata.file_name"))
    .withColumn("fecha_ingesta", current_timestamp())
)

# Escritura incremental en Delta
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", RUTA_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLA_DESTINO)
)

query.awaitTermination()

print("Ingesta Bronze de Devoluciones finalizada correctamente.")

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    MIN(fecha_ingesta) AS primera_ingesta,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM proyecto_final.bronze.devoluciones
GROUP BY archivo_origen
ORDER BY archivo_origen;